# Human grading: run agent on golden cases, grade each trajectory

In [7]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from agent.context import config_tag, load_prompt, max_rounds
from agent.judge import JUDGE_PROMPT, parse_json
from agent.llm import LLM
from agent.loop import investigate
from agent.schema import CRITERIA, Judgement
from agent.tools import LogTools, tool_purposes
from agent.trajectory import EventLog, Trajectory
from logstore.db import connect

GOLDEN_PATH = ROOT / "labeling" / "golden_cases.yaml"
GRADES_PATH = ROOT / "labeling" / "human_grades.csv"
TRAJ_DIR = ROOT / "labeling" / "trajectories"

CONFIG = config_tag()
CACHE_TAG = CONFIG

cases = yaml.safe_load(GOLDEN_PATH.read_text(encoding="utf-8"))
print(f"{len(cases)} golden cases: {[c['id'] for c in cases]}")
print(f"config: {CONFIG}  max_rounds: {max_rounds()}")

10 golden cases: ['c01', 'c02', 'c03', 'c05', 'c06', 'c07', 'c09', 'c10', 'c12', 'c17']
config: nemotron-3.5-lightning-30b-a3b_think200_8r  max_rounds: 8


## Run agent on all cases in parallel

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def run_one(case):
    conn = connect()
    tools = LogTools(conn)
    found = tools.logs.find_by_line(case["log_line"])
    assert len(found) == 1, f"{case['id']}: {len(found)} logs match"
    log = found[0]
    tools.visible = log.ts
    request_id = investigate(log, tools, LLM(), EventLog(), cache_tag=CACHE_TAG)
    return case["id"], request_id


with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(run_one, c): c["id"] for c in cases}
    results = {}
    for future in as_completed(futures):
        case_id, request_id = future.result()
        results[case_id] = request_id
        t = Trajectory(request_id)
        print(f"{case_id}: {request_id}  status={t.status}  rounds={t.totals['rounds']}")

print(f"\ndone: {len(results)} investigations")

c01: 7f252922b2f2  status=round_limit  rounds=8
c03: 541dcac9b180  status=ok  rounds=5
c06: aa080ecd3d9a  status=ok  rounds=5
c02: 8672d7227206  status=invalid_verdict  rounds=8
c05: 9d942818d7bf  status=ok  rounds=7
c07: 8e287a4b232c  status=invalid_verdict  rounds=8
c09: 482a45cd7088  status=invalid_verdict  rounds=8
c12: 0e1613fea4b4  status=invalid_verdict  rounds=8
c10: a82d0a618214  status=invalid_verdict  rounds=8
c17: 923d4765b8d4  status=invalid_verdict  rounds=8

done: 10 investigations


## Save trajectories to text files

In [9]:
TRAJ_DIR.mkdir(parents=True, exist_ok=True)

for case_id, request_id in sorted(results.items()):
    path = TRAJ_DIR / f"{case_id}_{CONFIG}__{request_id}.txt"
    path.write_text(Trajectory(request_id).text(), encoding="utf-8")
    print(f"{case_id} -> {path.name}")

print(f"\n{len(results)} trajectories saved to {TRAJ_DIR.relative_to(ROOT)}")

c01 -> c01_nemotron-3.5-lightning-30b-a3b_think200_8r__7f252922b2f2.txt
c02 -> c02_nemotron-3.5-lightning-30b-a3b_think200_8r__8672d7227206.txt
c03 -> c03_nemotron-3.5-lightning-30b-a3b_think200_8r__541dcac9b180.txt
c05 -> c05_nemotron-3.5-lightning-30b-a3b_think200_8r__9d942818d7bf.txt
c06 -> c06_nemotron-3.5-lightning-30b-a3b_think200_8r__aa080ecd3d9a.txt
c07 -> c07_nemotron-3.5-lightning-30b-a3b_think200_8r__8e287a4b232c.txt
c09 -> c09_nemotron-3.5-lightning-30b-a3b_think200_8r__482a45cd7088.txt
c10 -> c10_nemotron-3.5-lightning-30b-a3b_think200_8r__a82d0a618214.txt
c12 -> c12_nemotron-3.5-lightning-30b-a3b_think200_8r__0e1613fea4b4.txt
c17 -> c17_nemotron-3.5-lightning-30b-a3b_think200_8r__923d4765b8d4.txt

10 trajectories saved to labeling\trajectories
